# Error Handling

Welcome! Rust doesn't have exceptions. Instead, it uses two main tools: `panic!` for unrecoverable errors ("something went very wrong") and `Result<T, E>` for recoverable errors ("this might fail, and the caller should decide what to do"). This design makes error handling explicit and forces you to think about failure paths — leading to more robust code.

## panic! vs Recoverable Errors

- **`panic!`**: Unrecoverable. The program stops. Use when continuing would be meaningless (e.g., out-of-bounds array access, invariant violated).
- **Recoverable errors**: Return `Result<T, E>` or `Option<T>`. The caller can handle the failure (retry, log, show a message, etc.).

In [ ]:
// panic!("Something went wrong!");  // Uncomment to see a panic

fn might_fail(x: i32) -> Result<i32, String> {
    if x >= 0 { Ok(x * 2) } else { Err("negative input".to_string()) }
}

match might_fail(5) {
    Ok(v) => println!("Success: {}", v),
    Err(e) => println!("Error: {}", e),
}

## Result&lt;T, E&gt; and Option&lt;T&gt;

- **`Result<T, E>`**: Either `Ok(value)` or `Err(error)`. Use for operations that can fail (I/O, parsing, network).
- **`Option<T>`**: Either `Some(value)` or `None`. Use when "no value" is a valid outcome (e.g., "key not in map").

In [ ]:
fn divide(a: f64, b: f64) -> Result<f64, String> {
    if b == 0.0 { Err("division by zero".into()) }
    else { Ok(a / b) }
}

fn first_even(v: &[i32]) -> Option<i32> {
    v.iter().find(|&&x| x % 2 == 0).copied()
}

println!("divide: {:?}", divide(10.0, 2.0));
println!("first_even: {:?}", first_even(&[1, 3, 4, 5]));

## unwrap and expect

- **`unwrap()`**: Returns the value if `Ok`/`Some`, **panics** if `Err`/`None`. Quick and dirty — use only in prototypes or when you're 100% sure.
- **`expect(msg)`**: Same as `unwrap`, but lets you provide a custom panic message. Slightly better for debugging.

In [ ]:
let some_val: Option<i32> = Some(42);
let v = some_val.unwrap();  // 42
println!("v = {}", v);

let ok_val: Result<i32, &str> = Ok(100);
let n = ok_val.expect("should always be Ok here");
println!("n = {}", n);
// let bad = Err("oops").unwrap();  // Would panic!

## The ? Operator

`?` is a shorthand for "if Err, return the error immediately." It only works in functions that return `Result` (or `Option`). It propagates errors up the call stack — no more manual `match` and `return Err` everywhere!

In [ ]:
fn step1() -> Result<i32, String> {
    Ok(1)
}

fn step2() -> Result<i32, String> {
    Err("step2 failed".into())
}

fn do_work() -> Result<i32, String> {
    let a = step1()?;  // If Err, return immediately
    let b = step2()?;  // This will return Err
    Ok(a + b)
}

match do_work() {
    Ok(v) => println!("Result: {}", v),
    Err(e) => println!("Error: {}", e),
}

## map, and_then, unwrap_or

These methods let you transform and handle `Result` and `Option` without explicit `match`:
- **`map`**: Transform the inner value (if Ok/Some)
- **`and_then`**: Chain operations that return Result/Option
- **`unwrap_or`**: Provide a default if Err/None

In [ ]:
let opt: Option<i32> = Some(5);
let doubled = opt.map(|x| x * 2);
println!("doubled: {:?}", doubled);

let none: Option<i32> = None;
let with_default = none.unwrap_or(0);
println!("with_default: {}", with_default);

let res: Result<i32, &str> = Ok(10);
let squared = res.map(|x| x * x);
println!("squared: {:?}", squared);

In [ ]:
fn parse_and_double(s: &str) -> Option<i32> {
    s.parse::<i32>().ok().and_then(|n| Some(n * 2))
}

println!("{:?}", parse_and_double("21"));
println!("{:?}", parse_and_double("abc"));

## Custom Error Types

For real applications, define your own error type. It can hold context (what failed, where) and implement `std::error::Error` for compatibility with the ecosystem.

In [ ]:
use std::fmt;

#[derive(Debug)]
enum AppError {
    InvalidInput(String),
    Overflow,
}

impl fmt::Display for AppError {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        match self {
            AppError::InvalidInput(s) => write!(f, "Invalid input: {}", s),
            AppError::Overflow => write!(f, "Overflow occurred"),
        }
    }
}

impl std::error::Error for AppError {}

fn validate(x: i32) -> Result<i32, AppError> {
    if x < 0 { return Err(AppError::InvalidInput("must be non-negative".into())); }
    if x > 1000 { return Err(AppError::Overflow); }
    Ok(x)
}

println!("{:?}", validate(50));
println!("{:?}", validate(-1));

## From Trait for Error Conversion

Implementing `From<E> for YourError` lets you use `?` to automatically convert other error types into yours. Then `other_error?` becomes `Err(YourError::from(other_error))`.

In [ ]:
impl From<std::num::ParseIntError> for AppError {
    fn from(err: std::num::ParseIntError) -> Self {
        AppError::InvalidInput(err.to_string())
    }
}

fn parse_positive(s: &str) -> Result<i32, AppError> {
    let n: i32 = s.parse()?;  // ParseIntError converts to AppError
    if n < 0 { Err(AppError::InvalidInput("negative".into())) }
    else { Ok(n) }
}

println!("{:?}", parse_positive("42"));
println!("{:?}", parse_positive("abc"));

## thiserror and anyhow: Conceptual Overview

- **`thiserror`**: Derive macro for error types. Reduces boilerplate for `Display`, `Error`, and `From` implementations. Great for library authors defining error types.
- **`anyhow`**: For applications. `anyhow::Result<T>` = `Result<T, anyhow::Error>`. Easy to convert and chain errors, add context. Use in `main()` and application code.

Libraries: prefer custom errors (with thiserror). Applications: consider anyhow for flexibility.

## When to panic! vs Return Result

- **panic!**: Programming bugs (index out of bounds, unwrap on None), invariants that must never be violated, or when recovery is impossible (e.g., missing config file in a critical service).
- **Result**: User input errors, I/O failures, network timeouts, parsing errors — anything the caller might reasonably handle.

## Error Handling in main()

`main()` can return `Result<(), E>` where `E` implements `std::error::Error`. If you return `Err`, the program exits with a non-zero code and prints the error. This lets you use `?` in main!

In [ ]:
// In a real program, main could be:
// fn main() -> Result<(), Box<dyn std::error::Error>> {
//     let content = std::fs::read_to_string("file.txt")?;
//     println!("{}", content);
//     Ok(())
// }

fn run() -> Result<(), String> {
    let x = might_fail(10)?;
    println!("Got: {}", x);
    Ok(())
}

let _ = run();

## Best Practices

1. **Prefer `Result` over `panic!`** for recoverable failures
2. **Use `?` to propagate errors** — keeps code clean
3. **Add context** when propagating (e.g., `anyhow::Context`, or custom wrappers)
4. **Use `expect` with a helpful message** when you unwrap but want a clear panic
5. **Avoid `unwrap` in library code** — let callers decide how to handle errors
6. **Implement `Error` for custom errors** so they work with `?` and `Box<dyn Error>`

## Common Mistakes Beginners Make

1. **Overusing `unwrap()`** — leads to panics in production; use `?` or proper `match`
2. **Ignoring `Result`** — `let _ = result;` or forgetting to handle errors
3. **Using `panic!` for user errors** — return `Result` and let the caller handle it
4. **Mixing error types** — use `From` or `map_err` to unify error types in a function

## Key Rules to Remember

- `panic!` = unrecoverable; `Result`/`Option` = recoverable
- Use `?` to propagate errors in functions that return `Result`/`Option`
- `map`, `and_then`, `unwrap_or` help chain and transform Result/Option
- Custom errors: implement `Display` and `Error`; use `From` for conversion
- `main()` can return `Result<(), E>` to use `?` and exit with an error code
- Prefer returning `Result` in libraries; use `expect` sparingly with good messages